In [1]:
"""
Générateur de jeux de données TSPTW-D
======================================
Conforme au modèle du Livrable 1 (TSPTW-D) :
  - Graphe complet, coûts euclidiennes × scale → minutes
  - Fenêtres temporelles [a_i, b_i] cohérentes (instance réalisable)
  - Temps de service s_i
  - Perturbations dynamiques (arc, t_début, t_fin, alpha)

Format du fichier JSON généré
------------------------------
{
  "meta": { "n_clients": int, "scale": float, "seed": int },
  "depot": { "id": 0, "x": float, "y": float, "a": float, "b": float, "service": float },
  "clients": [
    { "id": int, "x": float, "y": float, "a": float, "b": float, "service": float },
    ...
  ],
  "perturbations": [
    { "arc": [i, j], "t_start": float, "t_end": float, "alpha": float },
    ...
  ]
}
"""

import json
import math
import random
from pathlib import Path


# ---------------------------------------------------------------------------
# Classe Ville (copie du projet pour pouvoir simuler la propagation)
# ---------------------------------------------------------------------------
class Ville:
    def __init__(self, nom: str, x: float, y: float,
                 fenetre: tuple = (0, float('inf')),
                 service: float = 0.0):
        self.nom     = nom
        self.x       = x
        self.y       = y
        self.a       = fenetre[0]
        self.b       = fenetre[1]
        self.service = service

    def __repr__(self):
        return self.nom


# ---------------------------------------------------------------------------
# Utilitaires
# ---------------------------------------------------------------------------

def _distance_minutes(v1: Ville, v2: Ville, scale: float) -> float:
    """Distance euclidienne convertie en minutes (c_ij^base)."""
    return math.hypot(v1.x - v2.x, v1.y - v2.y) * scale


def _propagation_greedy(villes: list[Ville], scale: float) -> float:
    """
    Simule une tournée dans l'ordre naturel (dépôt → v1 → v2 → … → dépôt)
    et retourne l'heure de retour au dépôt.
    Sert uniquement à calibrer b_0 (fenêtre du dépôt).
    """
    t = 0.0
    prev = villes[0]
    for ville in villes[1:]:
        t = max(t, ville.a) + ville.service
        t += _distance_minutes(prev, ville, scale)
        prev = ville
    t += _distance_minutes(prev, villes[0], scale)
    return t


# ---------------------------------------------------------------------------
# Fonction principale
# ---------------------------------------------------------------------------

def generer_dataset_tsptwd(
    n_clients: int,
    villes_raw: list[Ville],
    *,
    chemin_sortie: str | Path = "dataset_tsptwd.json",
    scale: float = 200.0,
    service_min: float = 5.0,
    service_max: float = 15.0,
    fenetre_largeur_min: float = 60.0,
    fenetre_largeur_max: float = 180.0,
    horizon: float = 480.0,
    n_perturbations: int | None = None,
    alpha_min: float = 1.5,
    alpha_max: float = 3.5,
    seed: int = 42,
) -> dict:
    """
    Génère un jeu de données TSPTW-D à partir de coordonnées brutes.

    Paramètres
    ----------
    n_clients       : nombre de clients souhaités (hors dépôt).
    villes_raw      : liste de Ville (coordonnées brutes), len >= n_clients + 1.
    chemin_sortie   : chemin du fichier JSON à écrire.
    scale           : facteur de conversion distance → minutes.
    service_min/max : plage du temps de service s_i (minutes).
    fenetre_largeur_min/max : plage de la largeur [a_i, b_i].
    horizon         : fin de journée (b_0 du dépôt, en minutes).
    n_perturbations : nombre de perturbations. Par défaut ≈ n_clients // 5.
    alpha_min/max   : plage du facteur de perturbation alpha.
    seed            : graine aléatoire (reproductibilité).

    Contraintes respectées
    ----------------------
    1. a_i < b_i  pour tout nœud.
    2. Les fenêtres sont compatibles avec un ordre de visite naturel
       (on génère a_i en simulant la propagation, garantissant qu'au
       moins une permutation est réalisable).
    3. b_0 = horizon (journée complète du dépôt).
    4. Les perturbations ne dépassent pas l'horizon.
    5. c_ij(t) > 0 pour tout t (alpha > 0 assuré).

    Retourne
    --------
    Le dictionnaire représentant le jeu de données (aussi écrit en JSON).
    """
    rng = random.Random(seed)

    if len(villes_raw) < n_clients + 1:
        raise ValueError(
            f"villes_raw contient seulement {len(villes_raw)} villes "
            f"mais n_clients+1 = {n_clients + 1} sont nécessaires."
        )

    # --- 1. Sélection des nœuds ------------------------------------------
    echantillon = villes_raw[:n_clients + 1]   # dépôt = index 0
    depot_raw   = echantillon[0]
    clients_raw = echantillon[1:]

    # --- 2. Dépôt -----------------------------------------------------------
    depot = Ville(
        nom     = "Dépôt",
        x       = float(depot_raw.x),
        y       = float(depot_raw.y),
        fenetre = (0.0, horizon),
        service = 0.0,
    )

    # --- 3. Fenêtres temporelles cohérentes pour les clients ---------------
    #
    # Principe : on simule une tournée greedy dans l'ordre d'indice pour
    # obtenir les heures d'arrivée "naturelles" tau_i, puis on construite
    # [a_i, b_i] autour de tau_i avec un écart aléatoire.
    # Cela garantit qu'il existe au moins une tournée réalisable.
    #
    clients: list[Ville] = []
    t = 0.0
    prev_x, prev_y = depot.x, depot.y

    for idx, v_raw in enumerate(clients_raw, start=1):
        # Temps de service
        s_i = round(rng.uniform(service_min, service_max), 1)

        # Distance depuis le nœud précédent → temps de transit
        dist = math.hypot(v_raw.x - prev_x, v_raw.y - prev_y) * scale
        t_arrivee = t + dist

        # Largeur de fenêtre
        largeur = round(rng.uniform(fenetre_largeur_min, fenetre_largeur_max), 1)

        # a_i : on ouvre la fenêtre un peu avant l'arrivée (ou à 0)
        marge_avant = round(rng.uniform(0, largeur * 0.4), 1)
        a_i = max(1.0, round(t_arrivee - marge_avant, 1))

        # b_i : a_i + largeur, mais borné à (horizon - s_i) pour laisser
        # du temps de service avant la fin de journée
        b_i = min(round(a_i + largeur, 1), horizon - s_i)

        # Sécurité : s'assurer que a_i < b_i
        if a_i >= b_i:
            b_i = a_i + fenetre_largeur_min

        client = Ville(
            nom     = f"Ville {idx}",
            x       = float(v_raw.x),
            y       = float(v_raw.y),
            fenetre = (a_i, b_i),
            service = s_i,
        )
        clients.append(client)

        # Mise à jour du temps courant pour la propagation greedy
        t = max(t_arrivee, a_i) + s_i
        prev_x, prev_y = v_raw.x, v_raw.y

    # --- 4. Perturbations ---------------------------------------------------
    if n_perturbations is None:
        n_perturbations = max(1, n_clients // 5)

    # Pool de tous les arcs possibles (indices 0..n_clients)
    tous_arcs = [
        (i, j)
        for i in range(n_clients + 1)
        for j in range(i + 1, n_clients + 1)
    ]
    arcs_choisis = rng.sample(tous_arcs, min(n_perturbations, len(tous_arcs)))

    perturbations = []
    for arc in arcs_choisis:
        t_start = round(rng.uniform(0, horizon * 0.6), 1)
        duree   = round(rng.uniform(30, horizon * 0.3), 1)
        t_end   = min(round(t_start + duree, 1), horizon)
        alpha   = round(rng.uniform(alpha_min, alpha_max), 2)
        perturbations.append({
            "arc":     list(arc),
            "t_start": t_start,
            "t_end":   t_end,
            "alpha":   alpha,
        })

    # --- 5. Construction du dictionnaire ------------------------------------
    dataset = {
        "meta": {
            "n_clients":   n_clients,
            "scale":       scale,
            "horizon":     horizon,
            "seed":        seed,
        },
        "depot": {
            "id":      0,
            "nom":     depot.nom,
            "x":       depot.x,
            "y":       depot.y,
            "a":       depot.a,
            "b":       depot.b,
            "service": depot.service,
        },
        "clients": [
            {
                "id":      i + 1,
                "nom":     c.nom,
                "x":       c.x,
                "y":       c.y,
                "a":       c.a,
                "b":       c.b,
                "service": c.service,
            }
            for i, c in enumerate(clients)
        ],
        "perturbations": perturbations,
    }

    # --- 6. Écriture du fichier JSON ----------------------------------------
    chemin_sortie = Path(chemin_sortie)
    chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
    with open(chemin_sortie, "w", encoding="utf-8") as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)

    print(f"[OK] Jeu de données généré : {chemin_sortie.resolve()}")
    print(f"     {n_clients} clients | {len(perturbations)} perturbation(s) | seed={seed}")

    return dataset


# ---------------------------------------------------------------------------
# Chargement d'un dataset depuis un fichier JSON
# ---------------------------------------------------------------------------

def charger_dataset_tsptwd(chemin: str | Path) -> tuple[list[Ville], list[dict]]:
    """
    Charge un fichier JSON généré par `generer_dataset_tsptwd`.

    Retourne
    --------
    villes       : liste[Ville] — index 0 = dépôt, 1..n = clients
    perturbations: list[dict]  — chaque dict : {arc, t_start, t_end, alpha}
    """
    with open(chemin, encoding="utf-8") as f:
        data = json.load(f)

    depot_d = data["depot"]
    depot = Ville(
        nom     = depot_d["nom"],
        x       = depot_d["x"],
        y       = depot_d["y"],
        fenetre = (depot_d["a"], depot_d["b"]),
        service = depot_d["service"],
    )

    clients = [
        Ville(
            nom     = c["nom"],
            x       = c["x"],
            y       = c["y"],
            fenetre = (c["a"], c["b"]),
            service = c["service"],
        )
        for c in data["clients"]
    ]

    perturbations = [
        {
            "arc":     tuple(p["arc"]),
            "t_start": p["t_start"],
            "t_end":   p["t_end"],
            "alpha":   p["alpha"],
        }
        for p in data["perturbations"]
    ]

    return [depot] + clients, perturbations


# ---------------------------------------------------------------------------
# Exemple d'utilisation
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Génération en lot — plusieurs tailles d'un coup
# ---------------------------------------------------------------------------

def _random_villes(n: int, seed: int = 0) -> list[Ville]:
    """Génère n villes aléatoires dans [0, 1]² (utilisé si villes_raw=None)."""
    rng = random.Random(seed)
    return [Ville(f"Raw {i}", rng.uniform(0, 1), rng.uniform(0, 1)) for i in range(n)]


def generer_batch_tsptwd(
    tailles: list[int],
    *,
    dossier_sortie: str | Path = "datasets",
    prefixe: str = "tsptwd_n",
    villes_raw: list[Ville] | None = None,
    perturbs_par_client: float = 0.1,
    seed: int = 42,
    **kwargs,
) -> dict[int, dict]:
    """
    Génère un fichier TSPTW-D pour chaque taille dans *tailles*.

    Paramètres
    ----------
    tailles             : liste de n_clients à générer, ex. [10, 50, 100, 200].
    dossier_sortie      : dossier de destination (créé si absent).
    prefixe             : préfixe du nom de fichier  → <prefixe><n>.json.
    villes_raw          : pool de villes brutes commun.  Si None, un pool
                          aléatoire de taille max(tailles)+1 est créé.
    perturbs_par_client : fraction de n utilisée pour n_perturbations
                          (ex. 0.1 → 10 % de n, minimum 1).
    seed                : graine globale (chaque taille dérive une sous-graine).
    **kwargs            : options transmises à generer_dataset_tsptwd()
                          (scale, horizon, service_min/max, …).

    Retourne
    --------
    dict {n: dataset_dict} pour chaque taille générée.
    """
    max_n = max(tailles)
    pool  = villes_raw if villes_raw is not None else _random_villes(max_n + 1, seed=seed)

    if len(pool) < max_n + 1:
        raise ValueError(
            f"villes_raw a {len(pool)} entrées mais la plus grande taille "            f"requiert {max_n + 1} nœuds (dépôt inclus)."        )

    resultats = {}
    for n in tailles:
        chemin = Path(dossier_sortie) / f"{prefixe}{n}.json"
        n_perturb = max(1, int(n * perturbs_par_client))
        sous_seed = seed + n          # seed reproductible par taille
        dataset = generer_dataset_tsptwd(
            n_clients       = n,
            villes_raw      = pool[:n + 1],
            chemin_sortie   = chemin,
            n_perturbations = n_perturb,
            seed            = sous_seed,
            **kwargs,
        )
        resultats[n] = dataset

    print(f"[BATCH TERMINÉ] {len(tailles)} fichier(s) dans '{dossier_sortie}/'")
    return resultats


# ---------------------------------------------------------------------------
# Exemple d'utilisation
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    generer_batch_tsptwd(
        tailles        = [5, 10, 50, 100, 200, 300, 500, 1000, 10000],
        dossier_sortie = "datasets",
        seed           = 42,
    )


[OK] Jeu de données généré : C:\Users\giaco\DEV\CESI\BLOC_RECHERCHE_OPERATIONNELLE\project\rec_op\datasets\tsptwd_n5.json
     5 clients | 1 perturbation(s) | seed=47
[OK] Jeu de données généré : C:\Users\giaco\DEV\CESI\BLOC_RECHERCHE_OPERATIONNELLE\project\rec_op\datasets\tsptwd_n10.json
     10 clients | 1 perturbation(s) | seed=52
[OK] Jeu de données généré : C:\Users\giaco\DEV\CESI\BLOC_RECHERCHE_OPERATIONNELLE\project\rec_op\datasets\tsptwd_n50.json
     50 clients | 5 perturbation(s) | seed=92
[OK] Jeu de données généré : C:\Users\giaco\DEV\CESI\BLOC_RECHERCHE_OPERATIONNELLE\project\rec_op\datasets\tsptwd_n100.json
     100 clients | 10 perturbation(s) | seed=142
[OK] Jeu de données généré : C:\Users\giaco\DEV\CESI\BLOC_RECHERCHE_OPERATIONNELLE\project\rec_op\datasets\tsptwd_n200.json
     200 clients | 20 perturbation(s) | seed=242
[OK] Jeu de données généré : C:\Users\giaco\DEV\CESI\BLOC_RECHERCHE_OPERATIONNELLE\project\rec_op\datasets\tsptwd_n300.json
     300 clients | 30 per

In [2]:
"""
Chargement d'une instance TSPTW-D en dictionnaire
==================================================
Structure retournée — prête pour l'optimisation :

instance = {
    # --- méta ---
    "n"      : int,           # nombre de clients (hors dépôt)
    "scale"  : float,
    "horizon": float,

    # --- nœuds (index 0 = dépôt, 1..n = clients) ---
    "coords"  : {i: (x, y)},  # coordonnées
    "a"       : {i: float},    # ouverture de fenêtre
    "b"       : {i: float},    # fermeture de fenêtre
    "service" : {i: float},    # temps de service

    # --- coûts de base (distance euclidienne × scale) ---
    "c_base"  : {(i,j): float},  # symétrique, i ≠ j

    # --- perturbations ---
    "perturbations": [
        {"arc": (i,j), "t_start": float, "t_end": float, "alpha": float},
        ...
    ],
}

Fonctions exposées
------------------
charger_instance(chemin)        → dict  (chargement complet)
c_ij(instance, i, j, t)        → float (coût dynamique à l'instant t)
"""

import json
import math
from pathlib import Path


# ---------------------------------------------------------------------------
# Chargement principal
# ---------------------------------------------------------------------------

def charger_instance(chemin: str | Path) -> dict:
    """
    Charge un fichier JSON généré par `generer_dataset_tsptwd`
    et retourne un dictionnaire prêt pour l'optimisation.

    Paramètre
    ---------
    chemin : chemin vers le fichier .json

    Retourne
    --------
    instance : dict avec les clés décrites en en-tête de module.
    """
    with open(chemin, encoding="utf-8") as f:
        data = json.load(f)

    n       = data["meta"]["n_clients"]
    scale   = data["meta"]["scale"]
    horizon = data["meta"]["horizon"]

    # --- nœuds : dépôt (0) + clients (1..n) --------------------------------
    coords  = {}
    a       = {}
    b       = {}
    service = {}

    d = data["depot"]
    coords[0]  = (d["x"], d["y"])
    a[0]       = d["a"]
    b[0]       = d["b"]
    service[0] = d["service"]

    for c in data["clients"]:
        i          = c["id"]          # 1..n
        coords[i]  = (c["x"], c["y"])
        a[i]       = c["a"]
        b[i]       = c["b"]
        service[i] = c["service"]

    # --- coûts de base : distance euclidienne × scale -----------------------
    noeuds = list(range(n + 1))
    c_base = {
        (i, j): round(
            math.hypot(coords[i][0] - coords[j][0],
                       coords[i][1] - coords[j][1]) * scale,
            4,
        )
        for i in noeuds
        for j in noeuds
        if i != j
    }

    # --- perturbations ------------------------------------------------------
    perturbations = [
        {
            "arc":     tuple(p["arc"]),
            "t_start": p["t_start"],
            "t_end":   p["t_end"],
            "alpha":   p["alpha"],
        }
        for p in data.get("perturbations", [])
    ]

    instance = {
        "n":             n,
        "scale":         scale,
        "horizon":       horizon,
        "coords":        coords,
        "a":             a,
        "b":             b,
        "service":       service,
        "c_base":        c_base,
        "perturbations": perturbations,
    }

    return instance


# ---------------------------------------------------------------------------
# Coût dynamique (modèle TSPTW-D du Livrable 1)
# ---------------------------------------------------------------------------

def c_ij(instance: dict, i: int, j: int, t: float) -> float:
    """
    Coût de transit dynamique de i vers j en partant à l'instant t.

        c_ij(t) = c_base[i,j] × (1 + δ_ij(t))

    δ_ij(t) = alpha - 1  si une perturbation sur (i,j) ou (j,i) est active à t
    δ_ij(t) = 0          sinon

    Paramètres
    ----------
    instance : dict retourné par charger_instance()
    i, j     : indices des nœuds (0 = dépôt)
    t        : instant de départ (minutes)

    Retourne
    --------
    Coût en minutes (float > 0).
    """
    base = instance["c_base"][(i, j)]
    for p in instance["perturbations"]:
        arc = p["arc"]
        if arc == (i, j) or arc == (j, i):   # symétrique
            if p["t_start"] <= t <= p["t_end"]:
                return base * p["alpha"]
    return base


# ---------------------------------------------------------------------------
# Propagation temporelle (utilitaire pour les algorithmes)
# ---------------------------------------------------------------------------

def evaluer_tournee(instance: dict, permutation: list[int],
                    dynamique: bool = True) -> tuple[bool, float]:
    """
    Évalue une permutation de clients et retourne (réalisable, coût_total).

    La permutation ne contient que les indices clients (1..n).
    Le dépôt (0) est ajouté automatiquement en début et fin.

        τ_{k+1} = max(τ_k, a_k) + service_k + c_ij(t_départ)

    Paramètres
    ----------
    instance    : dict retourné par charger_instance()
    permutation : liste d'indices clients, ex. [3, 1, 4, 2]
    dynamique   : si True, utilise c_ij() (coûts avec perturbations)
                  si False, utilise c_base (coûts statiques)

    Retourne
    --------
    (True,  coût)        si la tournée est réalisable
    (False, float('inf')) sinon
    """
    a       = instance["a"]
    b       = instance["b"]
    service = instance["service"]
    c_base  = instance["c_base"]

    route = [0] + list(permutation) + [0]
    t     = 0.0
    cout  = 0.0

    for k in range(len(route) - 1):
        i, j = route[k], route[k + 1]

        # Fenêtre de départ du nœud i
        if t > b[i]:
            return False, float("inf")

        # Attente éventuelle + service
        t = max(t, a[i]) + service[i]

        # Transit
        transit = c_ij(instance, i, j, t) if dynamique else c_base[(i, j)]
        cout   += transit
        t      += transit

    # Vérification retour au dépôt
    if t > b[0]:
        return False, float("inf")

    return True, round(cout, 4)


# ---------------------------------------------------------------------------
# Affichage récapitulatif
# ---------------------------------------------------------------------------

def afficher_instance(instance: dict) -> None:
    """Affiche un résumé lisible de l'instance."""
    n = instance["n"]
    print(f"Instance TSPTW-D")
    print(f"  Clients       : {n}")
    print(f"  Horizon       : {instance['horizon']} min")
    print(f"  Scale         : {instance['scale']}")
    print(f"  Perturbations : {len(instance['perturbations'])}")
    print()
    print(f"  {'Nœud':<10} {'Coords':>20}  {'[a, b]':>20}  {'service':>8}")
    print("  " + "-" * 65)
    for i in range(n + 1):
        nom  = "Dépôt" if i == 0 else f"Client {i}"
        x, y = instance["coords"][i]
        ai, bi = instance["a"][i], instance["b"][i]
        si = instance["service"][i]
        print(f"  {nom:<10} ({x:7.3f}, {y:7.3f})  [{ai:6.1f}, {bi:6.1f}]  {si:7.1f} min")

    if instance["perturbations"]:
        print()
        print(f"  {'Arc':<10} {'t_start':>10} {'t_end':>10} {'alpha':>8}")
        print("  " + "-" * 45)
        for p in instance["perturbations"]:
            print(f"  {str(p['arc']):<10} {p['t_start']:>10.1f} {p['t_end']:>10.1f} {p['alpha']:>8.2f}")


# ---------------------------------------------------------------------------
# Exemple d'utilisation
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    # Remplace le chemin par ton fichier généré
    instance = charger_instance("datasets/tsptwd_n20.json")

    afficher_instance(instance)

    # Accès direct aux structures utiles pour un algo
    n      = instance["n"]
    a      = instance["a"]       # ouvertures
    b      = instance["b"]       # fermetures
    s      = instance["service"] # temps de service
    c_base = instance["c_base"]  # {(i,j): float}

    # Coût dynamique entre nœuds 1 et 2 à t=50 min
    print(f"\nc(1,2) à t=50 min : {c_ij(instance, 1, 2, 50):.2f} min")

    # Évaluation d'une tournée exemple
    perm = list(range(1, n + 1))   # ordre naturel
    ok, cout = evaluer_tournee(instance, perm, dynamique=True)
    print(f"Tournée ordre naturel — réalisable: {ok}, coût: {cout:.1f} min")

Instance TSPTW-D
  Clients       : 10
  Horizon       : 480.0 min
  Scale         : 200.0
  Perturbations : 2

  Nœud                     Coords                [a, b]   service
  -----------------------------------------------------------------
  Dépôt      (  0.844,   0.758)  [   0.0,  480.0]      0.0 min
  Client 1   (  0.421,   0.259)  [ 124.0,  187.0]     11.4 min
  Client 2   (  0.511,   0.405)  [ 136.5,  284.9]      7.2 min
  Client 3   (  0.784,   0.303)  [ 230.2,  300.6]     13.9 min
  Client 4   (  0.477,   0.583)  [ 321.7,  407.9]      5.3 min
  Client 5   (  0.908,   0.505)  [ 410.4,  474.7]      5.3 min
  Client 6   (  0.282,   0.756)  [ 552.0,  612.0]     10.4 min
  Client 7   (  0.618,   0.251)  [ 684.6,  744.6]     13.1 min
  Client 8   (  0.910,   0.983)  [ 868.7,  928.7]     12.0 min
  Client 9   (  0.810,   0.902)  [ 908.9,  968.9]     14.6 min
  Client 10  (  0.310,   0.730)  [ 994.0, 1054.0]      6.0 min

  Arc           t_start      t_end    alpha
  ---------------